In [1]:
!pip install -q transformers datasets accelerate scikit-learn pandas matplotlib seaborn


In [2]:
from google.colab import files
uploaded = files.upload()


Saving arsentiment_test.csv to arsentiment_test.csv
Saving arsentiment_train.csv to arsentiment_train.csv
Saving arsentiment_val.csv to arsentiment_val.csv


In [ ]:
import os
import re
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    precision_recall_fscore_support,
    confusion_matrix,
    ConfusionMatrixDisplay
)

from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback
)

# =========================
# 1) الإعدادات
# =========================
DATA_PATH = "arsentiment_train.csv"
MODEL_NAME = "aubmindlab/bert-base-arabertv02-twitter"
OUTPUT_DIR = "./arabic_bert_sentiment_model"

RANDOM_STATE = 42
MAX_LENGTH = 128
TEST_SIZE = 0.15
VAL_SIZE = 0.15
EPOCHS = 5
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 16
LEARNING_RATE = 2e-5

# إذا مشروعك 3 فئات فقط
USE_THREE_CLASSES_ONLY = True

# =========================
# 2) تنظيف النص العربي
# =========================
def normalize_arabic(text):
    text = str(text).strip()

    # حذف الروابط والمنشن
    text = re.sub(r"http\S+|www\S+|https\S+", " ", text)
    text = re.sub(r"@\w+", " ", text)

    # حذف التشكيل
    text = re.sub(r"[\u0617-\u061A\u064B-\u0652]", "", text)

    # توحيد الحروف
    text = re.sub(r"[إأآا]", "ا", text)
    text = re.sub(r"ى", "ي", text)
    text = re.sub(r"ؤ", "و", text)
    text = re.sub(r"ئ", "ي", text)
    text = re.sub(r"ة", "ه", text)

    # حذف التطويل
    text = re.sub(r"ـ+", "", text)

    # تقليل تكرار الحروف
    text = re.sub(r"(.)\1{2,}", r"\1", text)

    # حذف الرموز غير المهمة
    text = re.sub(r"[^\u0600-\u06FFa-zA-Z0-9\s]", " ", text)

    # حذف المسافات الزائدة
    text = re.sub(r"\s+", " ", text).strip()

    return text

# =========================
# 3) قراءة البيانات
# =========================
df = pd.read_csv(DATA_PATH)

print("Columns:", df.columns.tolist())
print("Rows before cleaning:", len(df))
print(df.head())

required_cols = {"text", "sentiment"}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df = df[["text", "sentiment"]].dropna().copy()
df["text"] = df["text"].astype(str).str.strip()
df["sentiment"] = df["sentiment"].astype(str).str.strip()

# حذف النصوص الفارغة
df = df[df["text"] != ""].copy()

# توحيد أسماء الليبلات
df["sentiment"] = df["sentiment"].replace({
    "negative": "Negative",
    "neutral": "Neutral",
    "positive": "Positive",
    "NEGATIVE": "Negative",
    "NEUTRAL": "Neutral",
    "POSITIVE": "Positive",
    "neg": "Negative",
    "neu": "Neutral",
    "pos": "Positive"
})

# تنظيف النص العربي
df["text"] = df["text"].apply(normalize_arabic)

# حذف النصوص الفارغة بعد التنظيف
df = df[df["text"].str.len() > 0].copy()

# إذا بدك فقط 3 فئات
if USE_THREE_CLASSES_ONLY:
    df = df[df["sentiment"].isin(["Negative", "Neutral", "Positive"])].copy()

df = df.reset_index(drop=True)

print("\nRows after cleaning:", len(df))
print("\nClass distribution:")
print(df["sentiment"].value_counts())

# =========================
# 4) تشفير الليبلات
# =========================
label_encoder = LabelEncoder()
df["label"] = label_encoder.fit_transform(df["sentiment"])

label_names = list(label_encoder.classes_)
num_labels = len(label_names)

id2label = {i: label for i, label in enumerate(label_names)}
label2id = {label: i for i, label in enumerate(label_names)}

print("\nLabels:", label_names)

# =========================
# 5) تقسيم البيانات: train / val / test
# =========================
train_df, temp_df = train_test_split(
    df,
    test_size=(TEST_SIZE + VAL_SIZE),
    random_state=RANDOM_STATE,
    stratify=df["label"]
)

relative_val_size = VAL_SIZE / (TEST_SIZE + VAL_SIZE)

val_df, test_df = train_test_split(
    temp_df,
    test_size=(1 - relative_val_size),
    random_state=RANDOM_STATE,
    stratify=temp_df["label"]
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("\nTraining samples:", len(train_df))
print("Validation samples:", len(val_df))
print("Testing samples:", len(test_df))

# =========================
# 6) تحويل إلى Hugging Face Dataset
# =========================
dataset = DatasetDict({
    "train": Dataset.from_pandas(train_df[["text", "label"]], preserve_index=False),
    "validation": Dataset.from_pandas(val_df[["text", "label"]], preserve_index=False),
    "test": Dataset.from_pandas(test_df[["text", "label"]], preserve_index=False),
})

# =========================
# 7) تحميل tokenizer
# =========================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_LENGTH
    )

tokenized_dataset = dataset.map(tokenize_function, batched=True)

# =========================
# 8) Data collator
# =========================
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# =========================
# 9) تحميل النموذج
# =========================
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

# =========================
# 10) المقاييس
# =========================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    acc = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        preds,
        average="weighted",
        zero_division=0
    )

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

# =========================
# 11) إعدادات التدريب
# =========================
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    warmup_ratio=0.1,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=2,
    report_to="none",
    fp16=torch.cuda.is_available()
)

# =========================
# 12) Trainer
# =========================
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

# =========================
# 13) التدريب
# =========================
train_result = trainer.train()

# =========================
# 14) التقييم على validation
# =========================
val_metrics = trainer.evaluate(tokenized_dataset["validation"])
print("\nValidation evaluation:")
print(val_metrics)

# =========================
# 15) حساب Train Accuracy
# =========================
train_output = trainer.predict(tokenized_dataset["train"])
y_train_pred = np.argmax(train_output.predictions, axis=1)
y_train_true = train_output.label_ids
train_acc = accuracy_score(y_train_true, y_train_pred)

# =========================
# 16) حساب Test Accuracy
# =========================
test_output = trainer.predict(tokenized_dataset["test"])
y_test_pred = np.argmax(test_output.predictions, axis=1)
y_test_true = test_output.label_ids
test_acc = accuracy_score(y_test_true, y_test_pred)

print("\n" + "=" * 40)
print("ACCURACY RESULTS")
print("=" * 40)
print("Training Accuracy:", round(train_acc, 4))
print("Testing Accuracy :", round(test_acc, 4))

# =========================
# 17) Classification Report
# =========================
print("\nClassification Report (Test):\n")
print(classification_report(
    y_test_true,
    y_test_pred,
    target_names=label_names,
    zero_division=0
))

# =========================
# 18) رسم Confusion Matrix
# =========================
cm = confusion_matrix(y_test_true, y_test_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_names)

fig, ax = plt.subplots(figsize=(6, 6))
disp.plot(cmap="Blues", values_format="d", ax=ax)
plt.title("Confusion Matrix - Test Set")
plt.show()

# =========================
# 19) حفظ النموذج محليًا
# =========================
final_save_path = os.path.join(OUTPUT_DIR, "final_model")
os.makedirs(final_save_path, exist_ok=True)

trainer.model.save_pretrained(final_save_path)
tokenizer.save_pretrained(final_save_path)
joblib.dump(label_encoder, os.path.join(final_save_path, "sentiment_ar_label_encoder.pkl"))

print("\nSaved locally:")
print(final_save_path)
print(os.path.join(final_save_path, "sentiment_ar_label_encoder.pkl"))

# =========================
# 20) حفظ على Google Drive (اختياري)
# =========================
SAVE_TO_DRIVE = True

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

    drive_save_path = "/content/drive/MyDrive/CoCare/models/sentiment_ar_model"
    os.makedirs(drive_save_path, exist_ok=True)

    trainer.model.save_pretrained(drive_save_path)
    tokenizer.save_pretrained(drive_save_path)
    joblib.dump(label_encoder, os.path.join(drive_save_path, "sentiment_ar_label_encoder.pkl"))

    print("\nSaved to Google Drive:")
    print(drive_save_path)
    print(os.path.join(drive_save_path, "sentiment_ar_label_encoder.pkl"))

Columns: ['id', 'text', 'sentiment']
Rows before cleaning: 14054
         id                                               text sentiment
0  78975168  عاجل| وزارة الصحة: استشهاد اثنين بقصف صهيوني ش...  Negative
1  58359432  وزارة الصحة: "ارتقاء شاب متأثراً بإصابته برصاص...  Negative
2  70036334              بس مافهمت ليش كتبتو يتبع اكو جزء ثاني   Neutral
3  58364024  وزارة الصحة:  يجب ارتداء الكمامات للوقاية من ا...   Neutral
4  50481889  شركات #الإنترنت فى #قطاع_غزة  بتتصل فيه بتقله ...  Negative

Rows after cleaning: 13263

Class distribution:
sentiment
Negative    5539
Positive    5329
Neutral     2395
Name: count, dtype: int64

Labels: ['Negative', 'Neutral', 'Positive']

Training samples: 9284
Validation samples: 1989
Testing samples: 1990


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/667 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/476 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.528502,0.362179,0.864253,0.865184,0.864253,0.863762
2,0.265633,0.394216,0.873806,0.874558,0.873806,0.870499


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


In [ ]:
from sklearn.metrics import accuracy_score
import numpy as np

# ===== Testing Accuracy =====
test_output = trainer.predict(tokenized_dataset["test"])

y_test_pred = np.argmax(test_output.predictions, axis=1)
y_test_true = test_output.label_ids

test_acc = accuracy_score(y_test_true, y_test_pred)

print("=" * 40)
print("TEST ACCURACY")
print("=" * 40)
print("Testing Accuracy:", round(test_acc, 4))

In [ ]:
import os

print(os.listdir("/content/drive/MyDrive/CoCare/models/sentiment_ar_model"))

In [ ]:
import os
import joblib

base = "/content/drive/MyDrive/CoCare/models"
sentiment_path = os.path.join(base, "sentiment_ar_model")
sentiment_label_path = os.path.join(base, "sentiment_ar_label_encoder.pkl")

os.makedirs(sentiment_path, exist_ok=True)

trainer.model.save_pretrained(sentiment_path)
tokenizer.save_pretrained(sentiment_path)
joblib.dump(label_encoder, sentiment_label_path)
print("saved sentiment model to:", sentiment_path)
print("models folder:", os.listdir(base))
print("sentiment folder:", os.listdir(sentiment_path))
print("label encoder exists:", os.path.exists(sentiment_label_path))